In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Ontology Graph V4: Configuration-Driven Context Graph Pipeline

**Integrating BigQuery Agent Analytics SDK with YAML-Driven Property Graphs**

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ontology_graph_v4_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/main/examples/ontology_graph_v4_demo.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32"> Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ontology_graph_v4_demo.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35"> Open in BQ Studio
    </a>
  </td>
</table>

## Overview

The **Ontology Graph V4** pipeline introduces a fully configuration-driven approach to building
context graphs from unstructured ADK telemetry. Instead of hard-coding entity extraction logic,
the entire ontology is declared in a **YAML specification** file.

The pipeline performs the following steps:

1. **YAML Spec** defines the ontology: entities (with typed properties and keys), relationships
   (with source/target bindings), and table bindings.
2. **AI.GENERATE** (or Gemini API fallback) extracts a typed graph from unstructured ADK
   telemetry, guided by a compiled extraction prompt and output schema.
3. **Physical tables** are created and populated automatically based on the spec bindings.
4. **BigQuery Property Graph DDL** is generated dynamically from the spec and executed.
5. **GQL showcase queries** demonstrate graph traversal over the materialized property graph.

This notebook walks through each phase individually, then shows the one-shot orchestrator and
CLI commands that run the full pipeline in a single call.

## Demo Scenario: Yahoo ADCP Ad Decisioning

This demo uses the **Yahoo Ad Context Protocol (ADCP)** ad decisioning use case. The ontology
models the following domain objects and relationships:

```
DecisionPoint --CandidateEdge--> YahooAdUnit
RejectionReason --ForCandidate--> YahooAdUnit
```

- **DecisionPoint**: A point in the ad-serving pipeline where candidate ads are evaluated.
- **YahooAdUnit**: An ad creative that is a candidate for serving.
- **CandidateEdge**: Links a decision point to the ad units it considered.
- **RejectionReason**: Captures why a specific ad unit was rejected at a decision point.
- **ForCandidate**: Links a rejection reason back to the ad unit it pertains to.

The YAML spec (`ymgo_graph_spec.yaml`) encodes this ontology. AI.GENERATE extracts structured
graph data from raw ADK session logs and materializes it into BigQuery tables and a native
property graph.

## Install Dependencies

In [ ]:
!pip install -q bigquery-agent-analytics google-cloud-bigquery pyyaml

## Authenticate & Configure

In [ ]:
import os

try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab -- using default credentials.")

# ---------- Configuration ----------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")

## Phase 1: Load the YAML Ontology Spec

The YAML spec is the single source of truth for the ontology. It declares:

- **Entities** with typed properties, primary keys, and BigQuery table bindings.
- **Relationships** with source/target entity references and edge properties.

The `load_graph_spec` function parses the YAML file and resolves environment placeholders
(e.g., `{env}`) so that table references point to the correct BigQuery dataset.

In [ ]:
# Write the YMGO graph spec into the working directory so it is
# available regardless of how the notebook was launched (Colab,
# Vertex AI Workbench, local Jupyter, etc.).

SPEC_PATH = "ymgo_graph_spec.yaml"

_YMGO_SPEC = """\
# YMGO (Yahoo Media Graph Ontology) V3 — Context Graph Specification
#
# This YAML defines the ontology for a configuration-driven context graph.
# Use {{ env }} placeholders for environment-specific table bindings,
# resolved at load time via load_graph_spec(path, env="project.dataset").

graph:
  name: YMGO_Context_Graph_V3

  # ==========================================
  # 1. ENTITIES (Nodes)
  # ==========================================
  entities:
    - name: mako_DecisionPoint
      description: "The atomic unit of decisioning where an agent evaluates alternatives."
      binding:
        source: "{{ env }}.decision_points"
      keys:
        primary: [decision_id]
      properties:
        - name: decision_id
          type: string
        - name: decision_type
          type: string

    - name: sup_YahooAdUnit
      extends: mako_Candidate
      description: "A specific ad slot on a Yahoo property being evaluated as a candidate."
      binding:
        source: "{{ env }}.yahoo_ad_units"
      keys:
        primary: [adUnitId]
      properties:
        - name: adUnitId
          type: string
        - name: adUnitName
          type: string
        - name: adUnitSize
          type: string
          description: "e.g., '300x250', '728x90'"
        - name: adUnitPosition
          type: string
          description: "ATF (above the fold) | BTF (below the fold)"

    - name: mako_RejectionReason
      description: "Structured reason why a Candidate was not selected at a DecisionPoint."
      binding:
        source: "{{ env }}.rejection_reasons"
      keys:
        primary: [rejection_id]
      properties:
        - name: rejection_id
          type: string
        - name: rejectionType
          type: string
          description: "RULE_BASED | MODEL_BASED | TIMEOUT | ERROR"
        - name: rejectionRule
          type: string
          description: "The specific rule or model threshold that caused rejection."

  # ==========================================
  # 2. RELATIONSHIPS (Edges)
  # ==========================================
  relationships:
    - name: CandidateEdge
      description: "Connects a decision point to the evaluated Yahoo Ad Unit."
      from_entity: mako_DecisionPoint
      to_entity: sup_YahooAdUnit
      binding:
        source: "{{ env }}.candidate_edges"
        from_columns: [decision_id]
        to_columns: [adUnitId]
      properties:
        - name: edge_type
          type: string
          description: "SELECTED_CANDIDATE or DROPPED_CANDIDATE"
        - name: mako_scoreValue
          type: double
          description: "The confidence or predicted Q-value for this candidate."

    - name: ForCandidate
      description: "Maps the MAKO rejection rationale directly to the dropped candidate."
      from_entity: mako_RejectionReason
      to_entity: sup_YahooAdUnit
      binding:
        source: "{{ env }}.rejection_mappings"
        from_columns: [rejection_id]
        to_columns: [adUnitId]
"""

with open(SPEC_PATH, "w") as _f:
    _f.write(_YMGO_SPEC)

print(f"Wrote {SPEC_PATH} ({len(_YMGO_SPEC)} bytes)")


In [ ]:
from bigquery_agent_analytics.ontology_models import load_graph_spec

ENV = f"{PROJECT_ID}.{DATASET_ID}"

spec = load_graph_spec(SPEC_PATH, env=ENV)
print(f"Graph: {spec.name}")
print(f"Entities: {[e.name for e in spec.entities]}")
print(f"Relationships: {[r.name for r in spec.relationships]}")

## Inspect Entity Details

Each entity in the spec has a name, primary key columns, typed properties, and a binding
that maps it to a physical BigQuery table.

In [ ]:
for entity in spec.entities:
    print(f"Entity: {entity.name}")
    print(f"  Keys: {entity.keys}")
    print(f"  Properties: {[p.name for p in entity.properties]}")
    print(f"  Binding source: {entity.binding.source}")
    print()

## Phase 2: Compile the Extraction Schema

Before calling AI.GENERATE, we need two artifacts compiled from the spec:

1. **Extraction Prompt** -- a natural-language instruction that tells the LLM what entities
   and relationships to extract from the raw telemetry.
2. **Output Schema** -- a JSON schema that constrains the LLM output to match the declared
   ontology structure.

Both are generated deterministically from the YAML spec.

In [ ]:
from bigquery_agent_analytics.ontology_schema_compiler import (
    compile_extraction_prompt,
    compile_output_schema,
)

prompt = compile_extraction_prompt(spec)
print("=== Extraction Prompt ===")
print(prompt[:500])

schema = compile_output_schema(spec)
import json
print("\n=== Output Schema (truncated) ===")
print(json.dumps(schema, indent=2)[:600])

## Phase 3: Extract the Graph (AI.GENERATE)

The `OntologyGraphManager` orchestrates graph extraction. It reads raw ADK telemetry from
BigQuery, sends it through AI.GENERATE (or falls back to the Gemini API), and returns a
typed graph with nodes and edges.

Set `use_ai_generate=True` to use BigQuery ML's `AI.GENERATE` function, or `False` to use
the Gemini API directly.

In [ ]:
from bigquery_agent_analytics.ontology_graph import OntologyGraphManager

extractor = OntologyGraphManager(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    spec=spec,
    table_id=TABLE_ID,
    endpoint="gemini-2.5-flash",
)

SESSION_IDS = ["YOUR_SESSION_ID_HERE"]  # Replace with your session IDs

graph = extractor.extract_graph(
    session_ids=SESSION_IDS,
    use_ai_generate=True,
)
print(f"Extracted {len(graph.nodes)} nodes, {len(graph.edges)} edges")

In [ ]:
for node in graph.nodes[:5]:
    print(f"  [{node.entity_name}] {node.node_id}")
    for p in node.properties:
        print(f"    {p.name} = {p.value}")

## Phase 4: Create Tables & Materialize

The `OntologyMaterializer` uses the spec bindings to:

1. **Create physical BigQuery tables** for each entity and relationship.
2. **Route extracted nodes and edges** into the correct tables based on entity/relationship
   name matching.
3. **Insert rows** using the BigQuery streaming API.

In [ ]:
from bigquery_agent_analytics.ontology_materializer import OntologyMaterializer

materializer = OntologyMaterializer(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    spec=spec,
)

tables = materializer.create_tables()
print("Tables created:")
for name, ref in tables.items():
    print(f"  {name} -> {ref}")

rows = materializer.materialize(graph, SESSION_IDS)
print(f"\nRows materialized: {rows}")

## Phase 5: Generate Property Graph DDL

BigQuery supports native **Property Graphs** via DDL. The `OntologyPropertyGraphCompiler`
transpiles the YAML spec into a `CREATE PROPERTY GRAPH` statement that references the
physical tables created in Phase 4.

This step bridges the relational materialization with graph-native query capabilities (GQL).

In [ ]:
from bigquery_agent_analytics.ontology_property_graph import (
    OntologyPropertyGraphCompiler,
    compile_property_graph_ddl,
)

# Preview the DDL
ddl = compile_property_graph_ddl(spec, PROJECT_ID, DATASET_ID)
print(ddl)

In [ ]:
compiler = OntologyPropertyGraphCompiler(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    spec=spec,
)
created = compiler.create_property_graph()
print(f"Property Graph created: {created}")

## Phase 6: GQL Showcase Query

With the property graph in place, we can run **GQL (Graph Query Language)** queries to
traverse the graph. The SDK provides a `compile_showcase_gql` helper that generates a
representative traversal query from the spec.

In [ ]:
from bigquery_agent_analytics.ontology_orchestrator import compile_showcase_gql

gql = compile_showcase_gql(spec, PROJECT_ID, DATASET_ID)
print("=== GQL Showcase Query ===")
print(gql)

In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)
job = bq.query(
    gql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("session_id", "STRING", SESSION_IDS[0]),
            bigquery.ScalarQueryParameter("result_limit", "INT64", 50),
        ]
    ),
)
results = job.to_dataframe()
print(f"Rows returned: {len(results)}")
results.head(10)

## Phase 7: One-Shot Pipeline (`build_ontology_graph`)

The `build_ontology_graph` orchestrator runs the entire pipeline in a single call:
spec loading, extraction, table creation, materialization, property graph DDL, and
showcase GQL generation. This is the recommended entry point for production use.

In [ ]:
from bigquery_agent_analytics.ontology_orchestrator import build_ontology_graph

result = build_ontology_graph(
    session_ids=SESSION_IDS,
    spec_path=SPEC_PATH,
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    env=ENV,
)

print(f"Graph: {result['graph_name']}")
print(f"Graph ref: {result['graph_ref']}")
print(f"Nodes: {len(result['graph'].nodes)}")
print(f"Edges: {len(result['graph'].edges)}")
print(f"Tables: {list(result['tables_created'].keys())}")
print(f"Rows: {result['rows_materialized']}")
print(f"Property Graph created: {result['property_graph_created']}")

## Phase 8: CLI Commands

The same pipeline is available from the command line via `bq-agent-sdk`. This is
useful for CI/CD pipelines, cron jobs, or quick ad-hoc runs without a notebook.

In [ ]:
# Full pipeline
!bq-agent-sdk ontology-build \
    --project-id={PROJECT_ID} \
    --dataset-id={DATASET_ID} \
    --spec-path={SPEC_PATH} \
    --session-ids={','.join(SESSION_IDS)} \
    --env={ENV}

In [ ]:
# Generate GQL query
!bq-agent-sdk ontology-showcase-gql \
    --project-id={PROJECT_ID} \
    --dataset-id={DATASET_ID} \
    --spec-path={SPEC_PATH} \
    --env={ENV}

## Phase 9: Adapt for Your Own Ontology

The V4 pipeline is fully generic. To model your own domain:

1. **Copy** `ymgo_graph_spec.yaml` as a starting template.
2. **Replace entities** with your domain objects (e.g., `Customer`, `Order`, `Product`).
3. **Define relationships** between them (e.g., `Placed` from `Customer` to `Order`).
4. **Set `binding.source`** to point to your BigQuery tables.
5. **Run the same pipeline** -- no code changes required.

Below is a minimal 2-entity, 1-relationship example you can customize.

In [ ]:
CUSTOM_SPEC_YAML = """
graph:
  name: my_domain_graph

  entities:
    - name: Customer
      description: "A customer who places orders."
      binding:
        source: "{{ env }}.customers"
      keys:
        primary: [customer_id]
      properties:
        - name: customer_id
          type: string
        - name: customer_name
          type: string
        - name: email
          type: string

    - name: Order
      description: "A purchase order."
      binding:
        source: "{{ env }}.orders"
      keys:
        primary: [order_id]
      properties:
        - name: order_id
          type: string
        - name: order_date
          type: string
        - name: total_amount
          type: string

  relationships:
    - name: Placed
      description: "Customer placed an order."
      from_entity: Customer
      to_entity: Order
      binding:
        source: "{{ env }}.placed_edges"
        from_columns: [customer_id]
        to_columns: [order_id]
      properties:
        - name: placed_at
          type: string
"""

print(CUSTOM_SPEC_YAML)


In [ ]:
# Save the custom spec and run the pipeline
custom_spec_path = "my_domain_graph_spec.yaml"

with open(custom_spec_path, "w") as f:
    f.write(CUSTOM_SPEC_YAML)

print(f"Custom spec saved to: {custom_spec_path}")

# Load and verify
custom_spec = load_graph_spec(custom_spec_path, env=ENV)
print(f"Graph: {custom_spec.name}")
print(f"Entities: {[e.name for e in custom_spec.entities]}")
print(f"Relationships: {[r.name for r in custom_spec.relationships]}")

# Run the full pipeline with your custom spec
# result = build_ontology_graph(
#     session_ids=["YOUR_SESSION_ID"],
#     spec_path=custom_spec_path,
#     project_id=PROJECT_ID,
#     dataset_id=DATASET_ID,
#     env=ENV,
# )
# print(result)